In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import numpy as np

def expand_bbox(box, image_shape, expand_ratio=0.2):
    x1, y1, x2, y2 = map(int, box)
    h, w = image_shape[:2]
    bw = x2 - x1
    bh = y2 - y1
    dx = int(bw * expand_ratio)
    dy = int(bh * expand_ratio)
    x1_exp = max(0, x1 - dx)
    y1_exp = max(0, y1 - dy)
    x2_exp = min(w, x2 + dx)
    y2_exp = min(h, y2 + dy)
    return x1_exp, y1_exp, x2_exp, y2_exp

# Load image
image_path = "D:\\Master\\python-for-dl-homework\\week13\\redline_images\\1.png"
image_bgr = cv2.imread(image_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Load YOLOv8 large
model = YOLO("yolov8l.pt")
results = model(image_rgb)
boxes = results[0].boxes
class_names = model.names

# Crop and visualize
patches = []
labels = []
coords = []

for box in boxes:
    cls_id = int(box.cls[0])
    class_name = class_names[cls_id]
    if class_name in ["car", "motorcycle", "bus"]:
        x1, y1, x2, y2 = expand_bbox(box.xyxy[0], image_bgr.shape, expand_ratio=0.3)
        patch = image_rgb[y1:y2, x1:x2]
        patches.append(patch)
        labels.append(class_name)
        coords.append((x1, y1, x2, y2))

# Show cropped patches
plt.figure(figsize=(15, 5))
for i, (patch, label, (x1, y1, x2, y2)) in enumerate(zip(patches, labels, coords)):
    plt.subplot(1, len(patches), i + 1)
    plt.imshow(patch)
    plt.title(f"{label}\n({x1},{y1}) → ({x2},{y2})")
    plt.axis("off")
    
plt.tight_layout()
plt.show()